# Notebook 03 — Norms, Broadcasting, and Distances

**Estimated focused time:** 75–105 minutes. **Prerequisites:** Notebook 01 shapes and axes; Notebook 02 dot products and matrix operations.

## Learning objectives

After this notebook you should be able to (1) calculate and interpret L1/L2 norms and cosine similarity, (2) standardize feature columns with broadcasting, (3) compute a pairwise Euclidean distance matrix, and (4) explain how feature units affect nearest-neighbor results.

## Just enough theory

- The **L1 norm** adds absolute component values; the **L2 norm** is the square root of the sum of squared components. L2 responds more strongly to a large component.
- **Cosine similarity** compares direction using a dot product divided by both L2 norms. It is undefined when either vector has zero length; this notebook raises `ValueError` for that case.
- In an `n × d` feature matrix, subtracting a length-`d` vector broadcasts one value per feature across all `n` observations. Standardization subtracts each feature mean and divides by its population standard deviation. A constant column becomes zeros and uses a scale of `1` to avoid division by zero.
- **Euclidean distance** is the L2 norm of the difference between two rows. It depends on units; standardization can change which observations are nearest. Similarity on unlabeled transactions is **not** a fraud prediction.

Complete the starter code yourself. Self-check cells will fail until you implement the tasks. Use NumPy and Python only; the reference calculations in tests are for verification, not implementations to copy.

## Synthetic transaction data

Each row is one transaction; columns are amount, recent transaction count, account age in days, and merchant diversity. The data are small enough to inspect by hand and contain no fraud labels.

In [1]:
import numpy as np

X = np.array([[100., 1., 365., 5.],
              [110., 1., 370., 5.],
              [300., 8., 360., 2.],
              [105., 2., 30., 5.],
              [95., 1., 355., 6.]])
feature_names = ('amount', 'recent_count', 'account_age_days', 'merchant_diversity')
assert X.shape == (5, 4)

## Task 1 — Norms and cosine similarity (20–25 minutes)

**Goal:** Implement two vector-size measures and a direction comparison.

### Instructions

1. Implement `vector_norm(vector, order)` for a nonempty numeric 1D vector and `order` 1 or 2. Reject a matrix, empty input, or unsupported order with `ValueError`.
2. Implement `cosine_similarity(a, b)` for equal-length nonzero 1D vectors. Reject shape mismatch and a zero vector with `ValueError`.
3. Calculate the L1 and L2 norms of `[3, -4]` by hand; explain why cosine similarity is unchanged if one nonzero vector is multiplied by a positive scalar.

**Expected output:** Norms `7` and `5`; cosine values `1`, `0`, and `-1` for identical, orthogonal, and opposite vectors.

### Acceptance criteria

- [ ] Both functions return scalars and match the hand/reference cases.
- [ ] Relevant invalid inputs raise `ValueError`.
- [ ] The cosine explanation distinguishes direction from magnitude.

**Hint:** A dot product combines corresponding components; use the L2 norm to normalize its result.

**Interpretation:** Write 2–3 sentences explaining what the result means and one limitation or ML consequence. Add your answer beneath this paragraph.

In [33]:
import math

def vector_norm(vector, order):
    vector = np.asarray(vector)
    if order not in [1,2] or order is None or vector.ndim!=1:
        raise ValueError
    if order == 1:
        return sum([abs(x) for x in vector])
    elif order == 2:
        return math.sqrt(sum([pow(x,2) for x in vector]))
    # TODO: validate input, then calculate the requested norm without np.linalg.norm.
    # raise NotImplementedError

def cosine_similarity(a, b):
    return (np.dot(a,b))/(vector_norm(a,2)*vector_norm(b,2))
    # TODO: validate inputs and calculate a scalar using a dot product and your L2 norm.
    # raise NotImplementedError

### Self-check

Run this cell after completing the starter code. Keep the expected values and assertions intact.

In [34]:
sample = np.array([[3., -4.],
                   [1,2]])
sample_2 = np.array([3., -4.])

In [35]:
sample_2

array([ 3., -4.])

In [36]:
vector_norm(sample, 1)
cosine_similarity(e1, e1)

ValueError: 

In [37]:
sample = np.array([3., -4.])
np.testing.assert_allclose(vector_norm(sample, 1), 7.)
np.testing.assert_allclose(vector_norm(sample, 2), 5.)
np.testing.assert_allclose(vector_norm(sample, 2), np.linalg.norm(sample))
e1, e2 = np.array([1., 0.]), np.array([0., 1.])
np.testing.assert_allclose([cosine_similarity(e1, e1), cosine_similarity(e1, e2),
                            cosine_similarity(e1, -e1), cosine_similarity(5*e1, e1)],
                           [1., 0., -1., 1.], atol=1e-12)
for args in ((np.ones((1, 2)), 2), ([], 1), ([1, 2], 3)):
    np.testing.assert_raises(ValueError, vector_norm, *args)
for args in ((e1, np.zeros(2)), (e1, np.ones(3))):
    np.testing.assert_raises(ValueError, cosine_similarity, *args)

AssertionError: ValueError not raised by vector_norm

## Task 2 — Feature-wise broadcasting and scaling (20–25 minutes)

**Goal:** Use the correct axis and handle a constant feature safely.

### Instructions

1. Implement `standardize_columns(data)` for a nonempty 2D numeric array. Return `(scaled, means, scales)`, where summaries are length `d`; use population standard deviation (`ddof=0`).
2. For each constant column, return zeros in `scaled` and `1` in `scales`. Do not mutate the input.
3. Create `row_centered` by subtracting each row's mean. Compare its column means with the correctly standardized data and explain the axis mistake.

**Expected output:** A `(5,4)` standardized matrix with near-zero feature means and unit feature standard deviations; the wrong row-centered result has the same shape but different meaning.

### Acceptance criteria

- [ ] Output shapes, values, constant-column behavior, and input immutability pass the tests.
- [ ] 1D and empty inputs raise `ValueError`.
- [ ] The explanation identifies which axis represents features.

**Hint:** A length-4 vector can broadcast across five rows; subtracting a length-5 row summary answers a different question.

**Interpretation:** Write 2–3 sentences explaining what the result means and one limitation or ML consequence. Add your answer beneath this paragraph.

In [ ]:
def standardize_columns(data):
    # TODO: validate, calculate column summaries, and return scaled, means, scales.
    raise NotImplementedError

# TODO: compute row_centered without changing X.
row_centered = None

### Self-check

Run this cell after completing the starter code. Keep the expected values and assertions intact.

In [ ]:
original = X.copy()
Z, means, scales = standardize_columns(X)
assert Z.shape == X.shape and means.shape == scales.shape == (4,)
np.testing.assert_allclose(means, X.mean(axis=0))
np.testing.assert_allclose(scales, X.std(axis=0, ddof=0))
np.testing.assert_allclose(Z.mean(axis=0), 0, atol=1e-12)
np.testing.assert_allclose(Z.std(axis=0, ddof=0), 1, atol=1e-12)
np.testing.assert_array_equal(X, original)
with_constant = np.column_stack((X, np.ones(len(X))))
Zc, _, sc = standardize_columns(with_constant)
np.testing.assert_allclose(Zc[:, -1], 0)
np.testing.assert_allclose(sc[-1], 1)
assert np.all(np.isfinite(Zc))
assert row_centered is not None and row_centered.shape == X.shape
assert not np.allclose(row_centered.mean(axis=0), 0)
np.testing.assert_raises(ValueError, standardize_columns, X[0])
np.testing.assert_raises(ValueError, standardize_columns, np.empty((0, 4)))

## Task 3 — Pairwise Euclidean distances (20–25 minutes)

**Goal:** Use broadcasting to compare every observation with every other observation.

### Instructions

1. Implement `pairwise_l2(data)` for a nonempty numeric 2D array, returning an `n × n` distance matrix. Use broadcasting or nested row loops; avoid building a model or using a distance library.
2. Predict the shape of a broadcast row-difference array before coding. Check `[0,0]` and `[3,4]` by hand.
3. Explain why the distance matrix is symmetric and has a zero diagonal.

**Expected output:** For two reference points, `[[0,5],[5,0]]`; for `X`, a finite `(5,5)` matrix.

### Acceptance criteria

- [ ] Hand and NumPy references agree within tolerance.
- [ ] Symmetry, zero diagonal, and nonnegativity pass.
- [ ] Invalid 1D or empty data raises `ValueError`.

**Hint:** If you use broadcasting, adding singleton axes to the two copies of an `n × d` matrix produces `n × n × d` differences.

**Interpretation:** Write 2–3 sentences explaining what the result means and one limitation or ML consequence. Add your answer beneath this paragraph.

In [ ]:
def pairwise_l2(data):
    # TODO: validate 2D input and return pairwise Euclidean distances.
    raise NotImplementedError

### Self-check

Run this cell after completing the starter code. Keep the expected values and assertions intact.

In [ ]:
tiny = np.array([[0., 0.], [3., 4.]])
np.testing.assert_allclose(pairwise_l2(tiny), [[0., 5.], [5., 0.]])
D_raw = pairwise_l2(X)
assert D_raw.shape == (5, 5) and np.all(np.isfinite(D_raw))
np.testing.assert_allclose(D_raw, D_raw.T, atol=1e-12)
np.testing.assert_allclose(np.diag(D_raw), 0, atol=1e-12)
assert np.all(D_raw >= -1e-12)
np.testing.assert_allclose(D_raw[0, 1], np.linalg.norm(X[0] - X[1]))
np.testing.assert_allclose(pairwise_l2(X[:1]), [[0.]])
np.testing.assert_raises(ValueError, pairwise_l2, X[0])
np.testing.assert_raises(ValueError, pairwise_l2, np.empty((0, 4)))

## Task 4 — A small nearest-neighbor decision (20–25 minutes)

**Goal:** Turn a distance matrix into a reproducible comparison and interpret unit sensitivity.

### Instructions

1. Implement `nearest_neighbors(data, query_index, k)` using `pairwise_l2`. Return a 1D integer array of `k` other row indices, ordered by increasing distance; break ties by smaller index.
2. Find the three nearest rows to transaction 0 in raw `X` and standardized `Z`. Explain why the third neighbor changes.
3. Reject an invalid query index or `k` outside `1..n-1`. Describe why these neighbors do not imply fraud risk.

**Expected output:** Raw indices `[1,4,2]`; standardized indices `[1,4,3]` for the supplied data.

### Acceptance criteria

- [ ] Both rankings and tie-breaking pass tests.
- [ ] Invalid requests raise `ValueError`.
- [ ] The interpretation connects a change in ranking to feature units and does not claim fraud detection.

**Hint:** Sort by distance first, then by index. Exclude the query before choosing `k` results.

**Interpretation:** Write 2–3 sentences explaining what the result means and one limitation or ML consequence. Add your answer beneath this paragraph.

In [ ]:
def nearest_neighbors(data, query_index, k):
    # TODO: validate inputs, exclude the query row, and return sorted indices.
    raise NotImplementedError

### Self-check

Run this cell after completing the starter code. Keep the expected values and assertions intact.

In [ ]:
np.testing.assert_array_equal(nearest_neighbors(X, 0, 3), [1, 4, 2])
np.testing.assert_array_equal(nearest_neighbors(Z, 0, 3), [1, 4, 3])
np.testing.assert_array_equal(nearest_neighbors(np.array([[0.], [1.], [-1.]]), 0, 2), [1, 2])
for query, k in ((-1, 1), (len(X), 1), (0, 0), (0, len(X))):
    np.testing.assert_raises(ValueError, nearest_neighbors, X, query, k)

## Completion checklist

- [ ] Every starter cell is implemented; no `TODO`, `None` placeholder, or `NotImplementedError` remains.
- [ ] Every self-check passes, including invalid-input tests.
- [ ] I can explain the hand-check and the ML interpretation in my own words.
- [ ] **Restart Kernel → Run All** succeeds in order, with no unexpected errors.
- [ ] I saved the notebook and recorded a meaningful Git commit.

**Conceptual exit questions**

1. What do L1 and L2 emphasize differently?
2. Why is cosine similarity undefined for a zero vector?
3. Which axis is used for feature-wise standardization?
4. How can a valid numeric distance be misleading when features have different units?

**Final deliverable:** Norm, cosine, standardization, distance, and neighbor functions with passing checks and a short comparison of raw versus standardized rankings.